# 09 — Casting Armour (V3.1)

Armor enchantments that trigger spells, slayer bonuses, and effects when the
wearer is hit. The defender's armor fires back at the attacker.

**Sections:**
1. Setup & shared definitions
2. Spell-type armor (cast a spell back at the attacker)
3. Slayer-type armor (race-resistant damage bonus)
4. Effect-type armor (piercing, poison, drains)
5. Greater-type armor (tri-elemental, avenging)
6. Cursed vs normal comparison
7. ChanceOfEffect sweep

In [ ]:
import logging
import dataclasses
from pathlib import Path

from omega.config.armor_enchantments import ArmorEnchantment
from omega.config.combat_scripts import CombatScript
from omega.config.creature_types import CreatureType
from omega.config.enchantments import Enchantment
from omega.model.constants import (
    SKILLID_ANATOMY, SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_WRESTLING,
)
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, ParameterSweep, Scenario, Variable,
    WeaponSpec, run_scenario, run_sweep,
)
from omega.reporting.tables import comparison_table, summary_table, format_table_html
from omega.reporting.plots import (
    armor_enchantment_comparison, comparison_overlay, damage_histogram,
    damage_vs_parameter,
)
from omega.logging import setup_logging
from IPython.display import HTML

setup_logging(level=logging.ERROR)

SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)
RUN_KW = dict(shard=shard)

WEAPON = WeaponSpec(name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP)
ATTACKER = CombatantSpec(
    name="Warrior", is_npc=True,
    skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
    str_=100, dex_=100, int_=25,
    weapon=WEAPON,
    properties={"Type": "Undead"},
)
DEFENDER = CombatantSpec(
    name="Defender",
    skills={SKILLID_WRESTLING: 60},
    str_=50, dex_=50, int_=100, hp=500,
    armor=ArmorSpec(name="Plate", ar=30),
)
ITERATIONS = 200
BASE_SEED = 42
print("Setup complete.")

## 2. Spell-Type Armor Enchantments

Armor that casts a spell back at the attacker when hit. Uses `ArmorSpec.enchant_with()`
with the `ArmorEnchantment` enum. The spell fires based on `ChanceOfEffect`.

In [ ]:
SPELL_BASE = ArmorSpec(
    name="Enchanted Plate", ar=30,
    properties={"ChanceOfEffect": 75, "EffectCircle": 8},
)

spell_armors = {
    "Plain (no enchant)": ArmorSpec(name="Plate", ar=30),
    "C1: Bungling (Clumsy)": SPELL_BASE.enchant_with(ArmorEnchantment.OF_BUNGLING),
    "C3: Daemon's Breath": SPELL_BASE.enchant_with(ArmorEnchantment.OF_DAEMONS_BREATH),
    "C7: Hellfire": SPELL_BASE.enchant_with(ArmorEnchantment.OF_HELLFIRE),
}

spell_results = {}
for label, armor in spell_armors.items():
    defn = dataclasses.replace(DEFENDER, armor=armor)
    spell_results[label] = run_scenario(
        Scenario(attacker=ATTACKER, defender=defn, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = spell_results[label].damage_stats
    r = spell_results[label].ratios
    rate = f"  onhit={r.onhit_trigger_rate:.0%}" if r.onhit_trigger_rate > 0 else ""
    print(f"  {label:28s}  mean={ds.mean:6.2f}{rate}")

In [ ]:
armor_enchantment_comparison(spell_results, title="Spell Armor: Damage by Circle")

In [ ]:
rows = comparison_table(
    spell_results,
    stats=["mean", "mean_on_hit", "median", "p5", "p95", "hit_rate",
           "onhit_trigger_rate", "onhit_trigger_rate_on_hit"],
)
HTML(format_table_html(rows))

## 3. Slayer-Type Armor (Race-Resistant)

Undead Hunter armor halves damage from matching Undead attackers. Uses `CreatureType`
enum for the `ProtectedType` property.

In [ ]:
hunter_armor = ArmorSpec(name="Plate", ar=30).enchant_with(ArmorEnchantment.UNDEAD_HUNTER)

slayer_results = {
    "vs Undead (match)": run_scenario(
        Scenario(attacker=ATTACKER, defender=dataclasses.replace(DEFENDER, armor=hunter_armor),
                 iterations=ITERATIONS, base_seed=BASE_SEED), **RUN_KW),
    "vs Generic (no match)": run_scenario(
        Scenario(attacker=dataclasses.replace(ATTACKER, properties={"Type": "Daemon"}),
                 defender=dataclasses.replace(DEFENDER, armor=hunter_armor),
                 iterations=ITERATIONS, base_seed=BASE_SEED), **RUN_KW),
    "Plain armor": run_scenario(
        Scenario(attacker=ATTACKER, defender=DEFENDER,
                 iterations=ITERATIONS, base_seed=BASE_SEED), **RUN_KW),
}

for label, cell in slayer_results.items():
    ds = cell.damage_stats
    print(f"  {label:25s}  mean={ds.mean:6.2f}")

comparison_overlay(slayer_results, title="Undead Hunter Armor: Matching vs Non-Matching")

## 6. Cursed vs Normal

Cursed armor inverts the spell target — instead of hitting the attacker, the spell
hits the defender (the armor wearer). This is a significant balance difference.

In [ ]:
normal_armor = ArmorSpec(
    name="Plate", ar=30,
    properties={"ChanceOfEffect": 100, "EffectCircle": 8},
).enchant_with(ArmorEnchantment.OF_DAEMONS_BREATH)

cursed_armor = ArmorSpec(
    name="Cursed Plate", ar=30,
    properties={"ChanceOfEffect": 100, "EffectCircle": 8, "Cursed": 1},
).enchant_with(ArmorEnchantment.OF_DAEMONS_BREATH)

cursed_results = {
    "Normal (spell hits attacker)": run_scenario(
        Scenario(attacker=ATTACKER,
                 defender=dataclasses.replace(DEFENDER, armor=normal_armor),
                 iterations=ITERATIONS, base_seed=BASE_SEED), **RUN_KW),
    "Cursed (spell hits defender)": run_scenario(
        Scenario(attacker=ATTACKER,
                 defender=dataclasses.replace(DEFENDER, armor=cursed_armor),
                 iterations=ITERATIONS, base_seed=BASE_SEED), **RUN_KW),
}

for label, cell in cursed_results.items():
    ds = cell.damage_stats
    print(f"  {label:35s}  mean={ds.mean:6.2f}")

comparison_overlay(cursed_results, title="Cursed vs Normal: Daemon's Breath Armor")

## 7. ChanceOfEffect Sweep

How does the trigger probability affect effective damage? Sweep ChanceOfEffect
from 10% to 90% on a Fireball armor piece.

In [ ]:
sweep = ParameterSweep(
    scenario=Scenario(
        attacker=ATTACKER,
        defender=dataclasses.replace(
            DEFENDER,
            armor=ArmorSpec(
                name="Plate", ar=30,
                properties={"EffectCircle": 8},
            ).enchant_with(ArmorEnchantment.OF_DAEMONS_BREATH),
        ),
        iterations=100,
        base_seed=1234,
    ),
    variables=(
        Variable.from_range("defender", "armor.properties.ChanceOfEffect",
                            start=10, stop=90, step=10),
    ),
)

sweep_result = run_sweep(sweep, **RUN_KW)
print(f"{len(sweep_result.cells)} cells completed in {sweep_result.total_time:.1f}s")

damage_vs_parameter(
    sweep_result,
    "defender.armor.properties.ChanceOfEffect",
    title="Mean Damage vs ChanceOfEffect (Fireball Armor)",
)

In [ ]:
rows = summary_table(
    sweep_result,
    stats=["mean", "median", "p5", "p95", "hit_rate", "onhit_trigger_rate"],
)
HTML(format_table_html(rows))